# UC Governance Framework

Centralized governed tags, ABAC policies, structural enforcement, and audit logging for `dbx_maven_market`.

## 1. Governed Tag Taxonomy

Define account-level governed tags with enforced allowed values, then apply them to existing assets.

In [0]:
%sql
-- Create governed tags with enforced value lists
CREATE GOVERNED TAG classification
  DESCRIPTION 'Data classification level for access control'
  VALUES ('PII', 'Financial', 'Public');

CREATE GOVERNED TAG data_country
  DESCRIPTION 'Country of the data for row-level filtering'
  VALUES ('USA', 'Canada', 'Mexico');

CREATE GOVERNED TAG owner_team
  DESCRIPTION 'Team owning the data asset'
  VALUES ('Analytics', 'DataEngineering');

CREATE GOVERNED TAG data_sensitivity
  DESCRIPTION 'Data sensitivity level'
  VALUES ('low', 'medium', 'high', 'critical');

In [0]:
%sql
use catalog maven_catalog;
-- Tag PII columns on ingested tables for column masking
SET TAG ON COLUMN maven_catalog.bronze.raw_customers.customer_address classification = PII;
SET TAG ON COLUMN silver.silver_customers.customer_address classification = PII;

-- Tag dimension tables with data_country for ABAC row filtering
-- (pipeline_service_principals is in EXCEPT so the DLT pipeline owner stays exempt,
--  satisfying the MV constraint; non-pipeline users get filtered normally)
SET TAG ON TABLE gold.dim_stores data_country = USA;
SET TAG ON COLUMN gold.dim_stores.store_country data_country = USA;
SET TAG ON TABLE gold.dim_customers data_country = USA;
SET TAG ON COLUMN gold.dim_customers.customer_country data_country = USA;

-- Tag gold fact_sales with ownership
SET TAG ON TABLE gold.fact_sales owner_team = Analytics;

## 2. ABAC Dynamic Column Masking (Tag-Based)

A single catalog-level policy that auto-masks **any** column tagged `classification=PII` — no hardcoded column names.

In [0]:
%sql
-- Group exemptions (Admin, Data_Compliance, pipeline_service_principals) are
-- handled by the policy's EXCEPT clause — the UDF only implements the mask.
-- NOTE: is_account_group_member() evaluates differently inside a UDF on
-- serverless compute (always returns TRUE for pipeline_service_principals),
-- so exemption logic must stay in the EXCEPT clause, not here.
--
-- CREATE OR REPLACE replaces across parameter types, so only ONE overload
-- can exist at a time. Current PII-tagged columns are STRING (customer_address).
CREATE OR REPLACE FUNCTION audit.mask_pii_fn(val STRING)
RETURN '***MASKED_PII***';

In [0]:
%sql
CREATE OR REPLACE POLICY pii_column_mask
ON CATALOG maven_catalog
COMMENT 'Auto-mask any column tagged classification=PII for non-privileged users'
COLUMN MASK audit.mask_pii_fn
TO `account users` EXCEPT `Admin`, `Data_Compliance`, `pipeline_service_principals`
FOR TABLES
MATCH COLUMNS has_tag_value('classification', 'PII') AS pii_col
ON COLUMN pii_col;

## 3. ABAC Row Filter Policy (Tag-Based)

Restrict row visibility based on `data_country` tags. Regional managers see only rows matching their assigned country (USA, Canada, Mexico).

In [0]:
%sql
-- Session mapping table: maps user emails to authorized countries
CREATE OR REPLACE TABLE maven_catalog.audit.user_country_mapping (
  user_email STRING NOT NULL,
  assigned_country STRING NOT NULL
)
COMMENT 'Maps user login emails to authorized countries for ABAC row filtering';

INSERT INTO maven_catalog.audit.user_country_mapping VALUES
  ('kavyaag15@gmail.com', 'USA'),
  ('work.nish13nt@gmail.com', 'Canada'),
  ('jayshreepawar405@gmail.com', 'Mexico');

SELECT * FROM maven_catalog.audit.user_country_mapping;

In [0]:
%sql
-- Group exemptions (Admin, pipeline_service_principals) are handled by the
-- policy's EXCEPT clause — the UDF only implements the data filter.
-- NOTE: is_account_group_member() evaluates differently inside a UDF on
-- serverless compute, so exemption logic must stay in the EXCEPT clause.
CREATE OR REPLACE FUNCTION audit.filter_by_country_fn(store_country STRING)
RETURN EXISTS (
    SELECT 1 FROM audit.user_country_mapping
    WHERE user_email = current_user()
      AND assigned_country = store_country
);

In [0]:
%sql
CREATE OR REPLACE POLICY country_row_filter
ON CATALOG maven_catalog
COMMENT 'Restrict rows by data_country tag — users see only their assigned country (USA/Canada/Mexico)'
ROW FILTER audit.filter_by_country_fn
TO `account users` EXCEPT `Admin`, `pipeline_service_principals`
FOR TABLES
WHEN has_tag('data_country')
MATCH COLUMNS has_tag('data_country') AS country_col
USING COLUMNS (country_col);

## 4. Structural Enforcement — Gold Schema

Audit monitor that flags any gold table missing the required `owner_team` or `data_sensitivity` tags.

In [0]:
%sql
SELECT
  t.table_name,
  CASE WHEN ot.tag_value IS NULL THEN 'MISSING' ELSE ot.tag_value END AS owner_team_status,
  CASE WHEN ds.tag_value IS NULL THEN 'MISSING' ELSE ds.tag_value END AS data_sensitivity_status
FROM maven_catalog.information_schema.tables t
LEFT JOIN maven_catalog.information_schema.table_tags ot
  ON t.table_catalog = ot.catalog_name
  AND t.table_schema = ot.schema_name
  AND t.table_name = ot.table_name
  AND ot.tag_name = 'owner_team'
LEFT JOIN maven_catalog.information_schema.table_tags ds
  ON t.table_catalog = ds.catalog_name
  AND t.table_schema = ds.schema_name
  AND t.table_name = ds.table_name
  AND ds.tag_name = 'data_sensitivity'
WHERE t.table_schema = 'gold'
  AND t.table_catalog = 'dbx_maven_market'
  AND NOT t.table_name LIKE '__materialization%'
  AND NOT t.table_name LIKE 'event_log%'
  AND (ot.tag_value IS NULL OR ds.tag_value IS NULL)
ORDER BY t.table_name;

In [0]:
%sql
use catalog maven_catalog;
-- dim_customers: high sensitivity (contains PII address data)
SET TAG ON TABLE gold.dim_customers owner_team = Analytics;
SET TAG ON TABLE gold.dim_customers data_sensitivity = high;

-- dim_products: low sensitivity (product catalog)
SET TAG ON TABLE gold.dim_products owner_team = Analytics;
SET TAG ON TABLE gold.dim_products data_sensitivity = low;

-- dim_stores: low sensitivity (store reference data)
SET TAG ON TABLE gold.dim_stores owner_team = Analytics;
SET TAG ON TABLE gold.dim_stores data_sensitivity = low;

-- dim_calendar: low sensitivity (date reference data)
SET TAG ON TABLE gold.dim_calendar owner_team = DataEngineering;
SET TAG ON TABLE gold.dim_calendar data_sensitivity = low;

-- fact_sales: already has owner_team=Analytics, just needs sensitivity
SET TAG ON TABLE gold.fact_sales data_sensitivity = medium;

-- fact_returns: medium sensitivity (transaction data)
SET TAG ON TABLE gold.fact_returns owner_team = Analytics;
SET TAG ON TABLE gold.fact_returns data_sensitivity = medium;

-- fact_inventory_snapshot: medium sensitivity (inventory levels)
SET TAG ON TABLE gold.fact_inventory_snapshot owner_team = Analytics;
SET TAG ON TABLE gold.fact_inventory_snapshot data_sensitivity = medium;

In [0]:
%sql
select * from  silver.silver_customers;

In [0]:
%sql
SELECT 
    is_account_group_member('Admin') AS is_admin_capital,
    is_account_group_member('admin') AS is_admin_lower,
    is_account_group_member('Data_Compliance') AS is_data_compliance;

## 6. ABAC Verification Script (CI/CD)

Identity-agnostic configuration checks — verifies tags, policies, UDFs, and structural compliance exist as expected. Safe to run as **any** service principal.

> **Does it matter who runs this?** For **configuration checks** (tags exist, policies exist, UDFs exist) — **no**, any identity works. For **runtime behavior checks** (is data actually masked?) — **yes**, because ABAC policies evaluate `is_account_group_member()` against the invoking user. An `Admin` user is in the EXCEPT list and will always see unmasked data. To verify masking works, run the runtime assertion as a non-privileged service principal.

In [0]:
from dataclasses import dataclass

@dataclass
class CheckResult:
    name: str
    passed: bool
    detail: str

results: list[CheckResult] = []
CATALOG = "maven_catalog"

# ── 1. Governed tags are applied to PII columns ──
pii_cols = spark.sql(f"""
    SELECT schema_name, table_name, column_name
    FROM {CATALOG}.information_schema.column_tags
    WHERE tag_name = 'classification' AND tag_value = 'PII'
""").collect()

expected_pii = {
    ("bronze", "raw_customers", "customer_address"),
    ("silver", "silver_customers", "customer_address"),
}
actual_pii = {(r.schema_name, r.table_name, r.column_name) for r in pii_cols}
missing_pii = expected_pii - actual_pii
results.append(CheckResult(
    "PII column tags",
    len(missing_pii) == 0,
    f"Missing: {missing_pii}" if missing_pii else f"{len(expected_pii)} columns tagged"
))

# ── 2. Gold tables have required owner_team + data_sensitivity ──
non_compliant = spark.sql(f"""
    SELECT t.table_name
    FROM {CATALOG}.information_schema.tables t
    LEFT JOIN {CATALOG}.information_schema.table_tags ot
      ON t.table_catalog = ot.catalog_name AND t.table_schema = ot.schema_name
      AND t.table_name = ot.table_name AND ot.tag_name = 'owner_team'
    LEFT JOIN {CATALOG}.information_schema.table_tags ds
      ON t.table_catalog = ds.catalog_name AND t.table_schema = ds.schema_name
      AND t.table_name = ds.table_name AND ds.tag_name = 'data_sensitivity'
    WHERE t.table_schema = 'gold' AND t.table_catalog = '{CATALOG}'
      AND NOT t.table_name LIKE '__materialization%'
      AND NOT t.table_name LIKE 'event_log%'
      AND (ot.tag_value IS NULL OR ds.tag_value IS NULL)
""").collect()
results.append(CheckResult(
    "Gold structural compliance",
    len(non_compliant) == 0,
    f"Non-compliant: {[r.table_name for r in non_compliant]}" if non_compliant else "All gold tables tagged"
))

# ── 3. ABAC policies exist ──
policies = spark.sql(f"SHOW EFFECTIVE POLICIES ON CATALOG {CATALOG}").collect()
policy_names = {r["Policy Name"] for r in policies}

for expected in ["pii_column_mask", "country_row_filter"]:
    results.append(CheckResult(
        f"ABAC policy: {expected}",
        expected in policy_names,
        "Found" if expected in policy_names else "MISSING"
    ))

# ── 4. Mask and filter UDFs exist ──
for fn in ["mask_pii_fn", "filter_by_country_fn"]:
    try:
        spark.sql(f"DESCRIBE FUNCTION {CATALOG}.audit.{fn}").collect()
        results.append(CheckResult(f"UDF: {fn}", True, "Exists"))
    except Exception as e:
        results.append(CheckResult(f"UDF: {fn}", False, str(e)[:120]))

# ── 5. audit_logs table exists ──
try:
    cols = {r.col_name for r in spark.sql(f"DESCRIBE TABLE {CATALOG}.audit.audit_logs").collect()}
    required = {"log_id", "pipeline_name", "execution_status", "start_time", "log_timestamp"}
    missing = required - cols
    results.append(CheckResult(
        "audit_logs table",
        len(missing) == 0,
        f"Missing columns: {missing}" if missing else "Schema valid"
    ))
except Exception as e:
    results.append(CheckResult("audit_logs table", False, str(e)[:120]))

# ── 6. Runtime masking check (identity-sensitive!) ──
current_user = spark.sql("SELECT current_user()").collect()[0][0]
exempt_check = spark.sql("""
    SELECT
      is_account_group_member('Admin') OR
      is_account_group_member('Data_Compliance') OR
      is_account_group_member('pipeline_service_principals') AS is_exempt
""").collect()[0][0]
sample = spark.sql(f"""
    SELECT customer_address FROM {CATALOG}.silver.silver_customers LIMIT 1
""").collect()[0][0]

if exempt_check:
    # Exempt users (Admin, Data_Compliance, pipeline_service_principals) see real data
    is_visible = sample is not None and "MASKED" not in sample
    results.append(CheckResult(
        "Runtime: data visible to exempt user",
        is_visible,
        f"Exempt user ({current_user}) sees: '{str(sample)[:40]}'"
    ))
else:
    # Non-exempt should see masked/NULL data
    is_masked = sample is None or "MASKED" in sample
    results.append(CheckResult(
        "Runtime: masking enforced",
        is_masked,
        f"Non-exempt ({current_user}) sees: '{str(sample)[:40]}'"
    ))

# ── Report ──
print(f"{'CHECK':<35} {'RESULT':>6}  DETAIL")
print("─" * 90)
failed = 0
for r in results:
    status = "✅ PASS" if r.passed else "❌ FAIL"
    print(f"{r.name:<35} {status}  {r.detail}")
    if not r.passed:
        failed += 1

print(f"\n{'─' * 90}")
print(f"Total: {len(results)} checks | Passed: {len(results) - failed} | Failed: {failed}")

assert failed == 0, f"{failed} ABAC verification check(s) failed — see report above"

In [0]:
%sql
-- ─── Step 1: Who am I and which groups exempt me? ───
SELECT current_user() AS current_user,
       is_account_group_member('Admin') AS in_admin,
       is_account_group_member('Data_Compliance') AS in_data_compliance,
       is_account_group_member('pipeline_service_principals') AS in_psp;

-- ─── Step 2: Does the mask UDF return masked output? ───
-- The UDF always masks; the policy EXCEPT clause controls who is exempt
SELECT audit.mask_pii_fn('123 Secret Road') AS masked_string;

-- ─── Step 3: Does the row filter UDF correctly restrict by country? ───
-- Should be TRUE only for YOUR assigned country in user_country_mapping
SELECT
  audit.filter_by_country_fn('USA')    AS usa_allowed,
  audit.filter_by_country_fn('Canada') AS canada_allowed,
  audit.filter_by_country_fn('Mexico') AS mexico_allowed;

In [0]:
%sql
-- CLS test: exempt users (Admin / Data_Compliance / pipeline_service_principals)
-- see real addresses; everyone else sees '***MASKED_PII***'
SELECT customer_id, first_name, last_name, customer_address, customer_country
FROM maven_catalog.silver.silver_customers
LIMIT 5;

In [0]:
%sql
-- RLS test: exempt users see all 3 countries; non-exempt users see only their assigned one
SELECT 'dim_customers' AS table_name, customer_country, COUNT(*) AS row_count
FROM maven_catalog.gold.dim_customers
GROUP BY customer_country

UNION ALL

SELECT 'dim_stores', store_country, COUNT(*)
FROM maven_catalog.gold.dim_stores
GROUP BY store_country

ORDER BY table_name, customer_country;

## Expected Outputs by User

### User-to-Country Mapping (`audit.user_country_mapping`)
| User Email | Assigned Country |
| --- | --- |
| kavyaag15@gmail.com | USA |
| work.nish13nt@gmail.com | Canada |
| jayshreepawar405@gmail.com | Mexico |

### Column Masking (CLS) — `silver.silver_customers.customer_address`
| User | Group | Exempt? | `customer_address` value |
| --- | --- | --- | --- |
| kavyaag15@gmail.com | pipeline_service_principals | Yes | Real address (e.g. `2433 Bailey Road`) |
| work.nish13nt@gmail.com | (none) | No | `***MASKED_PII***` |
| jayshreepawar405@gmail.com | (none) | No | `***MASKED_PII***` |

### Row-Level Filtering (RLS) — `gold.dim_customers` / `gold.dim_stores`
| User | Group | Exempt? | Rows visible |
| --- | --- | --- | --- |
| kavyaag15@gmail.com | pipeline_service_principals | Yes | All countries (USA + Canada + Mexico) |
| work.nish13nt@gmail.com | (none) | No | Canada only |
| jayshreepawar405@gmail.com | (none) | No | Mexico only |

### UDF Direct-Call Results
| UDF Call | kavya (USA) | nishant (Canada) | jayshree (Mexico) |
| --- | --- | --- | --- |
| `filter_by_country_fn('USA')` | TRUE | FALSE | FALSE |
| `filter_by_country_fn('Canada')` | FALSE | TRUE | FALSE |
| `filter_by_country_fn('Mexico')` | FALSE | FALSE | TRUE |
| `mask_pii_fn('any string')` | `***MASKED_PII***` | `***MASKED_PII***` | `***MASKED_PII***` |

> **Note:** The mask UDF always returns the masked value. Exemption is handled by the policy's `EXCEPT` clause — exempt users never invoke the UDF at all.